# DCT Laboratory — Volume II, Chapter 11
## Distributionally Robust Enterprise Optimization
**Seed `26211`** · Companion to Chapter 11 and **AXIOM Lab 2.11** (module **AXIOM-11**) · Mirrored in `DCT_V2_Ch11_Lab.xlsx`

This laboratory reproduces **Worked Example 11.1, *How Much Should Meridian Trust Its Law?***, with the book's
declarations, and works the five steps of AXIOM Lab 2.11: read the records, price the family, try balls, set a radius
from a record, and moments only. The law, the levers and the floor are Chapter 9's. The new declarations are the
records behind the law: the treasury's two closures in twenty years and the engineering office's forty quarters of
deviations. Every law the records allow is simulated on Chapter 9's 100,000 paths, drawn from Volume I, Chapter 8's
stream with seed 8, and checked on the book's 100,000 fresh paths, seed 26209; the closures a higher rate adds come
from seeds 1108 and 26208 (Online Appendix Remark 11.B.1), and the records of 1,000 paths from seeds 900000–900199
(Chapter 9's sample-size study). The lab's own seed, 26211, draws only the simulated records of Panel 1's coverage
check. Every Meridian number below is the book's (Table 11.3, Figures 11.1–11.3, Online Appendix Example 11.B.2); the
final cell checks them. Money is in \$ million.

## The declarations (Worked Example 11.1, Step 1; Online Appendix Remark 11.B.1)

**Chapter 9's law, levers and floor.** Planned liquidity follows the Chapter 1 plan, whose draw dips to \$482 million
in quarter 7 of the program's 12; the board's floor is \$350 million. Execution and market deviations $E$ move it,
$E' = 0.6E + w$ with $w$ of spread \$50 million a quarter; funding windows close at random, 0.10 a year, each removing
a gamma-distributed amount (mean \$120 million, standard deviation \$50 million) that is never recovered. A standby
**line** $l$ covers up to $l$ of each closure, for fees of $0.03\,l$ over the program; a quarter of committed
**delay** $\phi$ makes the draw 4 index points shallower, at \$8 million. The board's appetite: at most a 5 percent
chance of a breach of the floor in some quarter. Chapter 9 kept it with Volume I's package, a \$170 million line and
two-thirds of a quarter of delay, \$10.44 million.

**The records behind the law.** The treasury's rate, 0.10 a year, is two closures in its twenty years of history. The
engineering office's spread, \$50 million a quarter, is the root-mean-square of 39 innovations from forty quarters of
deviations (the persistence, 0.6, is taken as known). Neither record pins its number down. The **records' family**
holds every law of Chapter 9's form with closures at most 0.315 a year and a spread at most \$61.6 million, the
records' one-sided 95 percent bounds (Proposition 11.2).

**How the family's laws are simulated (Remark 11.B.1).** Every law is simulated on Chapter 9's paths, coupled so that
the declared law is exactly Chapter 9's. A rate $\lambda' > 0.10$ adds the closures of an independent stream of rate
$\lambda' - 0.10$, drawn by thinning one extra stream of rate $0.40 - 0.10$ with uniforms, so that a higher rate only
adds closures; a rate $\lambda' < 0.10$ keeps each of the paths' own closures with chance $\lambda'/0.10$. The extra
closures and the uniforms come from seed 1108 for the book's paths and seed 26208 for the fresh ones: each quarter
draws the uniforms for the paths' own closures first, then the extra stream's counts, gamma sizes and uniforms. A
spread $c \times \$50$ million scales every deviation by $c$, because the deviations are linear in their innovations.

In [ ]:
import math
import numpy as np
from scipy import optimize, stats
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 26211        # the lab's seed: it draws only the simulated records of Panel 1's coverage check
BOOK_SEED = 8       # Volume I, Chapter 8's stream: the book's 100,000 paths (Online Appendix 1.B.4)
FRESH_SEED = 26209  # the book's 100,000 fresh paths (Chapter 9's lab seed)
EXTRA_SEED = {BOOK_SEED: 1108, FRESH_SEED: 26208}   # the family's extra closures and uniforms (Remark 11.B.1)

N, Q, YEARS = 100_000, 12, 3.0              # paths; quarters of the program; its years
# Chapter 9's law in USD billion, exactly as Volume I's engine draws it; every result is reported in $ million
FLOOR = 0.35                                # the board's floor
A_E, SD_W = 0.6, 0.05                       # E' = 0.6 E + w, w with spread $50m a quarter
LAM, J_MEAN, J_SD = 0.10, 0.12, 0.05        # closures a year; gamma sizes, mean $120m and sd $50m, never recovered
GAMMA = ((J_MEAN / J_SD) ** 2, J_SD ** 2 / J_MEAN)      # the sizes' shape and scale
LAM_MAX = 0.40                              # the extra stream's rate ceiling (Remark 11.B.1)
DEPTH0, DPQ = 28.0, 4.0                     # the draw's depth (index points); a quarter of delay: 4 points shallower
FEE, VQ, ALPHA = 0.03, 8.0, 0.05            # line fees over the program; a quarter of delay ($m); the appetite
MAX_DELAY = DEPTH0 / DPQ                    # 7 quarters remove the draw
LINES = np.arange(0.0, 400.01, 5.0)         # the line on a $5m grid up to $400m; the delay is continuous

def relax(x0, x_inf, k, t):
    return x_inf + (x0 - x_inf) * np.exp(-k * t)

def trough(depth, tau, t):
    return depth * (t / tau) * np.exp(1.0 - t / tau)

TQ = np.arange(1, Q + 1) / 4.0                          # the twelve quarters, in years
BASE = 0.62 * relax(100.0, 112.0, 0.35, TQ) / 100.0     # planned liquidity without the draw (index 100 = $0.62bn)
SHAPE = 0.62 * trough(1.0, 2.0, TQ) / 100.0            # the draw per index point of depth, deepest at two years
LIFT = 1000.0 * DPQ * SHAPE                            # $m of liquidity a quarter of delay adds, quarter by quarter
MARGIN = 1000.0 * (BASE - DEPTH0 * SHAPE - FLOOR)      # the plan's margin above the floor ($m)

def draws(seed, n=N):
    # Volume I's stream in its order: each quarter the execution shocks, the closure counts, then the sizes path by path
    rng = np.random.default_rng(seed)
    W, events = np.empty((n, Q)), []
    for k in range(Q):
        W[:, k] = rng.normal(0.0, SD_W, n)
        count = rng.poisson(LAM / 4.0, n)
        events.append((np.repeat(np.arange(n), count), rng.gamma(*GAMMA, int(count.sum()))))
    E, e = np.zeros((n, Q)), np.zeros(n)
    for k in range(Q):
        e = A_E * e + W[:, k]
        E[:, k] = e
    return E, events

class Paths:
    # The paths of one law. A line l ($m) covers up to l of each closure; phi quarters of delay make the draw 28 - 4 phi
    # deep. A path stays above the floor exactly when the draw is no deeper than its critical depth, so chances are
    # exact counts and the least delay meeting a chance is a quantile of the critical depths.
    def __init__(self, E, events, seed):
        self.E, self.events, self.seed, self.n = E, events, seed, E.shape[0]
        self._crit, self._sorted = {}, {}
    @classmethod
    def draw(cls, seed, n=N):
        return cls(*draws(seed, n), seed)
    def lost(self, line):                   # cumulative closures the line does not cover (USD bn), quarter by quarter
        J, j = np.zeros((self.n, Q)), np.zeros(self.n)
        for k, (path, size) in enumerate(self.events):
            j = j + np.bincount(path, np.maximum(size - line / 1000.0, 0.0), self.n)
            J[:, k] = j
        return J
    def crit(self, line):                   # each path's critical depth
        key = round(float(line), 6)
        if key not in self._crit:
            self._crit[key] = np.min((BASE[None, :] + self.E - self.lost(line) - FLOOR) / SHAPE[None, :], axis=1)
        return self._crit[key]
    def p(self, line, phi):                 # the chance of a breach in some quarter
        return float(np.mean(self.crit(line) < DEPTH0 - DPQ * phi))
    def delay_for(self, line, p=ALPHA):     # the least delay meeting the chance p (NaN: no delay suffices)
        key = round(float(line), 6)
        if key not in self._sorted:
            self._sorted[key] = np.sort(self.crit(line))
        d = min(DEPTH0, self._sorted[key][int(np.floor(p * self.n + 1e-9))])
        return np.nan if d < 0 else (DEPTH0 - d) / DPQ

def cost(line, phi):
    return FEE * line + VQ * phi

def cheapest(P, p=ALPHA, lines=LINES):
    # Chapter 9's design: the cheapest committed package keeping the chance p on P's paths
    best = None
    for line in lines:
        phi = P.delay_for(line, p)
        if not np.isnan(phi) and (best is None or cost(line, phi) < best[2]):
            best = (float(line), float(phi), float(cost(line, phi)))
    return best

_EXTRA = {}
def extra_stream(P):
    # Remark 11.B.1: each quarter, uniforms for the paths' own closures; then the extra stream at rate 0.40 - 0.10
    # (counts, gamma sizes, uniforms); seed 1108 on the book's paths, 26208 on the fresh paths
    if P.seed not in _EXTRA:
        rng = np.random.default_rng(EXTRA_SEED[P.seed])
        own, more = [], []
        for k in range(Q):
            path, size = P.events[k]
            own.append(rng.random(len(path)))
            count = rng.poisson((LAM_MAX - LAM) / 4.0, P.n)
            jj = np.repeat(np.arange(P.n), count)
            more.append((jj, rng.gamma(*GAMMA, len(jj)), rng.random(len(jj))))
        _EXTRA[P.seed] = (own, more)
    return _EXTRA[P.seed]

def law(P, lam=LAM, sigma=50.0):
    # The law with closures at lam a year and a spread of sigma ($m a quarter), on P's paths (Remark 11.B.1)
    c = sigma / 50.0
    if abs(lam - LAM) < 1e-15:
        return P if c == 1.0 else Paths(P.E * c, P.events, P.seed)
    own, more = extra_stream(P)
    events = []
    for k, (path, size) in enumerate(P.events):
        if lam < LAM:                       # thinning: keep each own closure with chance lam / 0.10
            keep = own[k] < lam / LAM
            events.append((path[keep], size[keep]))
        else:                               # superposition: add the extra closures whose uniform is below the share
            jj, sz, u = more[k]
            add = u < (lam - LAM) / (LAM_MAX - LAM)
            events.append((np.concatenate([path, jj[add]]), np.concatenate([size, sz[add]])))
    return Paths(P.E * c, events, P.seed)

def robust10(D, S):
    # Chapter 10, Proposition 10.1(ii): the cheapest package robust on deviations within D and one closure of at most S
    phi = max(0.0, float(np.max((D - MARGIN) / LIFT)))
    u = min(S, float(np.min(MARGIN - D + LIFT * phi)))
    return S - u, phi

BOOK, FRESH = Paths.draw(BOOK_SEED), Paths.draw(FRESH_SEED)
PKG = cheapest(BOOK)                        # Chapter 9: the cheapest package keeping 5 percent under the law
PKG_P, PKG_F = BOOK.p(*PKG[:2]), FRESH.p(*PKG[:2])
D10, S10 = 200.0 - 150.0 * 0.38, 300.0 - 180.0 * 0.38    # Chapter 10's max-min cut, 0.38
CH10 = robust10(D10, S10)
plan = 1000.0 * (BASE - DEPTH0 * SHAPE)
print(f"planned liquidity dips to {plan.min():.0f} in quarter {plan.argmin() + 1}; a quarter of delay lifts it by "
      f"{LIFT[plan.argmin()]:.1f} there")
print(f"Volume I's package (Chapter 9's design): line {PKG[0]:.0f}, delay {PKG[1]:.4f}, cost {PKG[2]:.2f}; "
      f"under the law {100 * PKG_P:.2f}% (fresh {100 * PKG_F:.2f}%)")
print(f"Chapter 10's max-min package (deviations within {D10:.1f}, a closure within {S10:.1f}): line {CH10[0]:.1f}, "
      f"delay {CH10[1]:.3f}, cost {cost(*CH10):.2f}; under the law {100 * BOOK.p(*CH10):.2f}%")

## Panel 1 — Read the records (Proposition 11.2(i)–(iii); Worked Example 11.1, Step 1; Online Appendix Example 11.B.2: rate at most 0.3148 a year; spread from \$42.27m to \$61.60m; Volume I's package from 4.47% to 6.35% by rate and from 1.03% to 17.87% by spread; 15.79% at the corner)

AXIOM Lab 2.11, step 1. With $n$ events in $T$ years, the rate is at most $\chi^2_{1-\beta}(2n + 2)/(2T)$ with
confidence $1 - \beta$ (Garwood's bound, Proposition 11.2(i)); with $n$ Gaussian innovations of root-mean-square $s$,
the spread is at most $s\sqrt{n/\chi^2_{\beta}(n)}$ and at least $s\sqrt{n/\chi^2_{1-\beta}(n)}$, each with confidence
$1 - \beta$ (part (ii)). Two closures in twenty years and 39 innovations of \$50 million give the records' bounds at
95 percent each, and 90 percent together by Bonferroni (part (iii)); a record twice as long, at the same estimates,
gives narrower ones. Volume I's package is then judged under the laws the records allow, one doubt at a time, and at
the corner, both bounds at once: more frequent closures only remove liquidity and a wider spread raises the chance of
a breach (checked across the family in Panel 2), so the family's worst law is its corner. Last, the lab's own draws
(seed 26211): 100,000 records simulated under the declared law, to see how often the bounds hold the true rate and
spread.

In [ ]:
def rate_bounds(n, years, conf=0.95):
    # Proposition 11.2(i): Garwood's one-sided bounds for a Poisson rate from n events in `years` years
    lo = stats.chi2.ppf(1.0 - conf, 2 * n) / (2.0 * years) if n > 0 else 0.0
    return lo, stats.chi2.ppf(conf, 2 * n + 2) / (2.0 * years)

def spread_bounds(df, s=50.0, conf=0.95):
    # Proposition 11.2(ii): one-sided bounds for a spread from df Gaussian innovations of root-mean-square s
    return s * math.sqrt(df / stats.chi2.ppf(conf, df)), s * math.sqrt(df / stats.chi2.ppf(1.0 - conf, df))

LAM_L, LAM_U = rate_bounds(2, 20.0)         # the treasury: two closures in twenty years
SIG_L, SIG_U = spread_bounds(39)            # the engineering office: forty quarters, 39 innovations
LAM_U2 = rate_bounds(4, 40.0)[1]            # twice the record: four closures in forty years
SIG_L2, SIG_U2 = spread_bounds(79)          # eighty quarters, 79 innovations
print(f"rate: two closures in twenty years -> estimate {2 / 20:.2f}; one-sided 95% bounds {LAM_L:.4f} and {LAM_U:.4f} a year")
print(f"spread: 39 innovations of $50m -> one-sided 95% bounds {SIG_L:.2f} and {SIG_U:.2f}")
print(f"together (Bonferroni): at least {100 * (1 - 0.05 - 0.05):.0f}% confidence")
print(f"twice the record (four in forty years; 79 innovations): rate at most {LAM_U2:.4f}; spread at most {SIG_U2:.2f} "
      f"(at least {SIG_L2:.2f})")

RATES = [0.0, LAM_L, 0.05, LAM, 0.15, 0.2, 0.25, LAM_U, 0.35]
SPREADS = [40.0, SIG_L, 45.0, 50.0, 55.0, 60.0, SIG_U, 65.0]
BY_RATE = [law(BOOK, lam).p(*PKG[:2]) for lam in RATES]
BY_SPREAD = [law(BOOK, LAM, s).p(*PKG[:2]) for s in SPREADS]
CORNER, CORNER_F = law(BOOK, LAM_U, SIG_U), law(FRESH, LAM_U, SIG_U)   # the records' worst law, on both sets of paths
P_CORNER, P_CORNER_F = CORNER.p(*PKG[:2]), CORNER_F.p(*PKG[:2])
print("\nVolume I's package under the laws the records allow (spread $50m):")
print("  rate a year  " + " ".join(f"{x:6.3f}" for x in RATES))
print("  breach (%)   " + " ".join(f"{100 * x:6.2f}" for x in BY_RATE))
print("and at the declared rate, 0.10:")
print("  spread ($m)  " + " ".join(f"{x:6.1f}" for x in SPREADS))
print("  breach (%)   " + " ".join(f"{100 * x:6.2f}" for x in BY_SPREAD))
print(f"at the corner (rate {LAM_U:.4f}, spread {SIG_U:.1f}): {100 * P_CORNER:.2f}% (fresh {100 * P_CORNER_F:.2f}%), "
      f"against {100 * PKG_P:.2f}% under the declared law")

# the lab's own draws: records simulated under the declared law (seed 26211)
rng = np.random.default_rng(SEED)
M = 100_000
n_closures = rng.poisson(LAM * 20.0, M)
rms = np.sqrt((rng.normal(0.0, 50.0, (M, 39)) ** 2).mean(axis=1))
rate_holds = stats.chi2.ppf(0.95, 2 * n_closures + 2) / 40.0 >= LAM
spread_holds = rms * math.sqrt(39 / stats.chi2.ppf(0.05, 39)) >= 50.0
COVER = dict(rate=float(rate_holds.mean()), spread=float(spread_holds.mean()), both=float((rate_holds & spread_holds).mean()))
print(f"\n{M:,} records simulated under the declared law (the lab's seed {SEED}): the rate's bound holds the true rate in "
      f"{100 * COVER['rate']:.1f}%, the spread's in {100 * COVER['spread']:.1f}%, both in {100 * COVER['both']:.1f}%")
print(f"  with no closure at all the rate's bound is {rate_bounds(0, 20.0)[1]:.3f}, above 0.10: Garwood's bound errs on the "
      f"safe side; at a true rate of 0.20 it holds with chance "
      f"{100 * stats.poisson.sf(0, 0.20 * 20):.1f}%, at 0.3148 with {100 * stats.poisson.sf(1, LAM_U * 20):.1f}%")

## Panel 2 — Price the family (Theorem 11.1(iv); Proposition 11.2(iii); Worked Example 11.1, Step 2; Figure 11.3(a): a \$200m line and 2.25 quarters, \$24.00m; \$11.64m for the closures' doubt alone and \$22.85m for the spread's; at most 5.0% on a grid of 49 laws; 0.9% under the declared law)

AXIOM Lab 2.11, step 2. For each declaration of doubt, the cheapest package that keeps 5 percent at its worst law is
Chapter 9's design at that law: the line on a \$5 million grid up to \$400 million and the delay continuous, the least
delay at each line being a quantile of the paths' critical depths. Four declarations: the closures' doubt alone
(rate 0.3148, spread \$50 million), the spread's doubt alone (rate 0.10, spread \$61.6 million), the records' family
(both bounds) and a record twice as long (rate 0.2288, spread \$57.6 million). Each package is checked on the fresh
paths at its law and judged under the declared law. The family's package is then checked across the family, on a grid
of 49 laws (seven rates from 0 to 0.315, seven spreads from \$42.3 to \$61.6 million). By Theorem 11.1(iv), a family
that holds the true law with 90 percent confidence makes its package's promise hold with 90 percent confidence.

In [ ]:
def family_case(lam, sigma):
    B, F = law(BOOK, lam, sigma), law(FRESH, lam, sigma)
    d = cheapest(B)
    return dict(lam=lam, sigma=sigma, p_pkg=B.p(*PKG[:2]), p_pkg_fresh=F.p(*PKG[:2]), design=d, p_fresh=F.p(*d[:2]),
                p_law=BOOK.p(*d[:2]), p_law_fresh=FRESH.p(*d[:2]))

CASES = {"rate": family_case(LAM_U, 50.0), "spread": family_case(LAM, SIG_U), "both": family_case(LAM_U, SIG_U),
         "longer": family_case(LAM_U2, SIG_U2)}
NAMES = {"rate": "closures' doubt", "spread": "spread's doubt", "both": "the records' family", "longer": "twice the record"}
print(f"{'declaration':22s} {'rate':>6s} {'spread':>6s} | {'Vol I pkg':>9s} {'fresh':>6s} | {'line':>5s} {'delay':>6s} "
      f"{'cost':>6s} {'fresh':>6s} | {'declared law':>12s} {'fresh':>6s}")
for key, c in CASES.items():
    l_, ph, co = c["design"]
    print(f"{NAMES[key]:22s} {c['lam']:6.4f} {c['sigma']:6.1f} | {100 * c['p_pkg']:8.2f}% {100 * c['p_pkg_fresh']:5.2f}% | "
          f"{l_:5.0f} {ph:6.3f} {co:6.2f} {100 * c['p_fresh']:5.2f}% | {100 * c['p_law']:11.2f}% {100 * c['p_law_fresh']:5.2f}%")
FAM = CASES["both"]["design"]
GRID = [(float(lam), float(s), law(BOOK, lam, s).p(*FAM[:2])) for lam in np.linspace(0.0, LAM_U, 7)
        for s in np.linspace(SIG_L, SIG_U, 7)]
worst_law = max(GRID, key=lambda g: g[2])
print(f"\nthe family's package on the grid of 49 laws: largest chance {100 * worst_law[2]:.2f}% at rate {worst_law[0]:.4f}, "
      f"spread {worst_law[1]:.1f} (the corner)")
print(f"the spread is what costs: the closures' doubt adds {CASES['rate']['design'][2] - PKG[2]:.2f} to Volume I's "
      f"{PKG[2]:.2f}; the spread's {CASES['spread']['design'][2] - PKG[2]:.2f}; both {FAM[2] - PKG[2]:.2f}")

sg = np.linspace(40.0, 65.0, 26)
curves = {(pk, lam): [100 * law(BOOK, lam, s).p(*pk[:2]) for s in sg] for pk in (PKG, FAM) for lam in (LAM, LAM_U)}
fig, ax = plt.subplots(figsize=(7.6, 3.6))
for pk, col, name in ((PKG, "#1F4E79", "Volume I's package"), (FAM, "#E65100", "the records' family's package")):
    ax.plot(sg, curves[(pk, LAM)], color=col, lw=1.8, label=f"{name}, closures 0.10 a year")
    ax.plot(sg, curves[(pk, LAM_U)], color=col, lw=1.8, ls="--", label=f"{name}, closures 0.315 a year")
ax.axhline(5.0, color="#999999", lw=0.8, ls=":")
for v, txt in ((50.0, "declared"), (SIG_U, "bound")):
    ax.axvline(v, color="#999999", lw=0.8, ls="--")
    ax.text(v + 0.3, 19.3, txt, fontsize=8, color="#616161", va="top")
ax.annotate(f"{100 * P_CORNER:.1f}% at the corner", xy=(SIG_U, 100 * P_CORNER), xytext=(52.5, 16.5), fontsize=8,
            arrowprops=dict(arrowstyle="-", lw=0.6, color="#333333"))
ax.set(xlim=(40, 65), ylim=(0, 20), xlabel="spread of deviations ($ million a quarter)", ylabel="chance of a breach (%)",
       title="Volume I's package breaches on 15.8% at the records' corner (seed 26211)")
ax.legend(frameon=False, fontsize=7.5, loc="upper left"); plt.tight_layout(); plt.show()

## Panel 3 — Try balls: divergence (Theorem 11.2; Figure 11.1(a); Worked Example 11.1, Step 3: radii 0.439, 0.603 and 1.042; Volume I's package at 8.36% in the ball of 0.01; 2.5% under the law, a \$175m line and 1.354 quarters, \$16.08m; 3 in a million at 0.44; 2 in 100 billion at 1.04)

AXIOM Lab 2.11, step 3, first part. A Kullback–Leibler ball of radius $d$ around the declared law holds every law that
reweights its futures with $\mathrm{KL}(Q\,\|\,P) \le d$. The worst chance over the ball of an event of chance $p$ is the
$q^* \ge p$ with $\mathrm{kl}(q^*\,\|\,p) = d$ (Theorem 11.2(ii)); so the ball keeps 5 percent exactly when the law
keeps the smaller chance $\alpha'$ with $\mathrm{kl}(0.05\,\|\,\alpha') = d$, and the ball's package is Chapter 9's design
at the appetite $\alpha'$ — no new paths are needed. A ball must hold the doubts it is meant for: by part (iii),
closures at $\lambda'$ instead of $\lambda$ over $H = 3$ years lie at $H(\lambda'\log(\lambda'/\lambda) - \lambda' +
\lambda)$, twelve innovations with their spread scaled by $c$ at $12(\log(1/c) + (c^2 - 1)/2)$, and the two add. The
levers' reach ends where the law's appetite falls below one path in 100,000.

In [ ]:
def kl(q, p):
    # Theorem 11.2(ii): kl(q || p), the divergence of a chance q from a chance p
    out = 0.0
    if q > 0:
        out += q * math.log(q / p)
    if q < 1:
        out += (1 - q) * math.log((1 - q) / (1 - p))
    return out

def worst_chance(p, d):
    # q*(p, d): the worst chance over the ball of radius d of an event of chance p under the law
    if d <= 0:
        return p
    if kl(1.0 - 1e-15, p) <= d:
        return 1.0
    return optimize.brentq(lambda q: kl(q, p) - d, p, 1.0 - 1e-15, xtol=1e-15)

def appetite(d, a=ALPHA):
    # alpha'(d): the chance the law must keep for every law in the ball to keep a
    if d <= 0:
        return a
    return optimize.brentq(lambda p: kl(a, p) - d, 1e-300, a, xtol=1e-300, rtol=1e-14)

def kl_rate(lam, lam0=LAM, years=YEARS):    # Theorem 11.2(iii): closures at lam instead of lam0 over the program
    return years * (lam * math.log(lam / lam0) - lam + lam0)

def kl_spread(c, n=Q):                      # n Gaussian innovations whose spread is scaled by c
    return n * (math.log(1.0 / c) + (c * c - 1.0) / 2.0)

D_RATE, D_SPREAD = kl_rate(LAM_U), kl_spread(SIG_U / 50.0)
RADII = {"closures' doubt": D_RATE, "spread's doubt": D_SPREAD, "both doubts": D_RATE + D_SPREAD}
print(f"closures at {LAM_U:.4f}: {YEARS:.0f} x ({LAM_U:.4f} log({LAM_U:.4f} / 0.10) - {LAM_U - LAM:.4f}) = {D_RATE:.3f}")
print(f"spread x {SIG_U / 50:.3f}: {Q} x (log(1 / {SIG_U / 50:.3f}) + ({SIG_U / 50:.3f}^2 - 1) / 2) = {D_SPREAD:.3f}")
for k, d in RADII.items():
    print(f"  {k:16s} radius {d:.3f}: to keep 5% over that ball, the law must keep {appetite(d):.3g}")
P_MIN = BOOK.p(400.0, MAX_DELAY)
D_LEVERS = kl(ALPHA, P_MIN)
print(f"a $400m line and 7 quarters of delay breach on {P_MIN * N:.0f} path of {N:,}: beyond the levers from "
      f"kl(0.05 || {P_MIN:g}) = {D_LEVERS:.3f}")

KL_WORST = {d: worst_chance(PKG_P, d) for d in (0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0)}
print("\nworst chance of Volume I's package over the ball (it keeps 5.00% under the law):")
print("  d          " + " ".join(f"{d:6g}" for d in KL_WORST))
print("  worst (%)  " + " ".join(f"{100 * q:6.2f}" for q in KL_WORST.values()))

KL_D = (0.0, 0.001, 0.002, 0.005, 0.01, 0.02, 0.03, 0.05, 0.075, 0.1, 0.15)
KL_DESIGNS = {}
print("\nkeeping 5% over the ball: d, the law's appetite, line, delay, cost; under the law (fresh)")
for d in KL_D:
    a = appetite(d)
    b = cheapest(BOOK, p=a)
    KL_DESIGNS[d] = dict(alpha=a, design=b, p=BOOK.p(*b[:2]), p_fresh=FRESH.p(*b[:2]))
    print(f"  {d:5g}  {100 * a:5.2f}%  {b[0]:4.0f} {b[1]:6.3f} {b[2]:6.2f}   {100 * KL_DESIGNS[d]['p']:5.2f}% "
          f"({100 * KL_DESIGNS[d]['p_fresh']:.2f}%)")
for d in (0.01, 0.03):
    b = KL_DESIGNS[d]["design"]
    print(f"the ball of {d}: its package at the records' corner breaches on {100 * CORNER.p(*b[:2]):.2f}% "
          f"(fresh {100 * CORNER_F.p(*b[:2]):.2f}%)")

ds = np.exp(np.linspace(np.log(1e-3), np.log(1.5), 200))
fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.loglog(ds, [100 * worst_chance(PKG_P, d) for d in ds], color="#B71C1C", lw=1.8,
          label="worst chance of Volume I's package over the ball")
ax.loglog(ds, [100 * appetite(d) for d in ds], color="#1F4E79", lw=1.8, label="chance the law must keep for the ball to keep 5%")
ax.axhline(5.0, color="#999999", lw=0.8, ls=":")
ax.axhline(1e-3, color="#999999", lw=0.8, ls=":")
ax.text(1.15e-3, 1.4e-3, "one path in 100,000: the levers' floor", fontsize=8, color="#616161")
for d, txt in zip(RADII.values(), ("closures", "spread", "both")):
    ax.axvline(d, color="#E65100", lw=0.9, ls="--")
    ax.text(d / 1.04, 1.5e-7, f"{txt} {d:.2f}", fontsize=7.5, color="#E65100", rotation=90, ha="right", va="bottom")
ax.set(xlim=(1e-3, 1.5), ylim=(1e-7, 300), xlabel="radius of the ball, d", ylabel="chance of a breach (%)",
       title="Keeping 5% over a ball of 0.01 means 2.5% under the law (seed 26211)")
ax.set_yticks([1e-6, 1e-4, 1e-2, 1, 100], ["0.000001", "0.0001", "0.01", "1", "100"])
ax.set_xticks([1e-3, 1e-2, 1e-1, 1], ["0.001", "0.01", "0.1", "1"])
ax.legend(frameon=False, fontsize=8, loc="lower left"); plt.tight_layout(); plt.show()

## Panel 4 — Try balls: Wasserstein (Theorem 11.3(ii)–(iii); Figure 11.2(a); Worked Example 11.1, Step 3: Volume I's package at 7.11% in the ball of \$0.1m; a \$165m line and 1.148 quarters, \$14.13m, 3.23% under the law; the spread's doubt at \$26.3m, asking \$525m of the thinnest paths, against \$121.9m within the levers' reach)

AXIOM Lab 2.11, step 3, second part. A Wasserstein ball moves futures instead of reweighting them. Let a path move by
shifting its deviations, its closures fixed, at a cost equal to the largest shift in any quarter: a path's distance
from a breach is then its **thinnest margin** above the floor over the twelve quarters. The worst chance over the ball
of radius $\varepsilon$ moves the paths nearest the floor first, a fractional knapsack with budget $N\varepsilon$
(Theorem 11.3(iii)); so the ball keeps 5 percent exactly when the thinnest 5 percent of paths average at least
$\varepsilon/0.05$ above the floor. A package for the ball needs, at each line, the least delay that gives them that
average: margins only rise with the delay, so a bisection on the delay finds it (34 halvings of the 7 quarters, as the
book's engine), evaluating at each step only the paths that can still be among the thinnest. Scaling every deviation
by $c$ moves each path by $(c - 1)$ times its largest deviation, so the ball that holds the spread's doubt has radius
$(c - 1)$ times the paths' average largest deviation (Online Appendix Example 11.B.2).

In [ ]:
def margin_parts(P, line):
    # each path's margin above the floor in each quarter with no delay ($m)
    return 1000.0 * (BASE[None, :] - DEPTH0 * SHAPE[None, :] + P.E - P.lost(line) - FLOOR)

def thinnest(A, phi):
    # a path's distance from a breach: its thinnest margin over the quarters ($m)
    return np.min(A + LIFT[None, :] * phi, axis=1)

def worst_chance_w(m, eps):
    # Theorem 11.3(iii): the share of paths a budget of N eps can push below the floor, nearest first
    d = np.sort(np.maximum(m, 0.0))
    n = len(d)
    cs = np.cumsum(d)
    j = int(np.searchsorted(cs, n * eps, side="right"))
    if j >= n:
        return 1.0
    frac = (n * eps - (cs[j - 1] if j > 0 else 0.0)) / d[j] if d[j] > 0 else 1.0
    return (j + min(frac, 1.0)) / n

def thin_mean(m, a=ALPHA):
    # the average margin (floored at zero) of the thinnest share a of the paths
    return float(np.mean(np.sort(np.maximum(m, 0.0))[:int(round(a * len(m)))]))

def least_delays(A, radii, p=ALPHA, steps=34):
    # For each radius: the least delay at which the K = p n thinnest paths' margins sum to at least n * radius,
    # by bisection on [0, 7] quarters; NaN if 7 quarters do not suffice. After each halving, only the paths whose
    # margin at the interval's low end is within the K-th thinnest margin at its high end can matter.
    n = A.shape[0]
    K = int(round(p * n))
    need = n * np.asarray(radii, dtype=float)
    def thin_sum(Ac, phis):                 # sum of the K thinnest (in increasing order), the K-th, all margins
        d = np.maximum(np.min(Ac[None, :, :] + LIFT[None, None, :] * np.asarray(phis)[:, None, None], axis=2), 0.0)
        part = np.partition(d, K - 1, axis=1)[:, :K]
        return np.cumsum(np.sort(part, axis=1), axis=1)[:, -1], part.max(axis=1), d
    (s0, s7), (_, v7), (d0, _) = thin_sum(A, [0.0, MAX_DELAY])
    out = np.where(s7 >= need, np.where(s0 >= need, 0.0, MAX_DELAY), np.nan)
    act = np.nonzero((s7 >= need) & (s0 < need))[0]
    if len(act) == 0:
        return out
    need = need[act]
    lo, hi, v_hi = np.zeros(len(act)), np.full(len(act), MAX_DELAY), np.full(len(act), v7)
    m_lo = np.repeat(d0[None, :], len(act), axis=0)
    keep = (m_lo <= v_hi[:, None]).any(axis=0)
    Ac, m_lo = A[keep], m_lo[:, keep]
    for _ in range(steps):
        mid = 0.5 * (lo + hi)
        u, back = np.unique(mid, return_inverse=True)   # radii still sharing an interval share the work
        s, v, d = (x[back] for x in thin_sum(Ac, u))
        ok = s >= need
        hi, v_hi = np.where(ok, mid, hi), np.where(ok, v, v_hi)
        lo, m_lo = np.where(ok, lo, mid), np.where(ok[:, None], m_lo, d)
        keep = (m_lo <= v_hi[:, None]).any(axis=0)
        Ac, m_lo = Ac[keep], m_lo[:, keep]
    out[act] = hi
    return out

def design_w(P, radii, p=ALPHA, lines=np.arange(100.0, 300.01, 5.0), steps=34):
    # the cheapest package keeping p over the Wasserstein ball of each radius
    best = [None] * len(radii)
    for line in lines:
        for r, phi in enumerate(least_delays(margin_parts(P, line), radii, p, steps)):
            if not np.isnan(phi) and (best[r] is None or cost(line, phi) < best[r][2]):
                best[r] = (float(line), float(phi), float(cost(line, phi)))
    return best

M_PKG = thinnest(margin_parts(BOOK, PKG[0]), PKG[1])
W_WORST = {e: worst_chance_w(M_PKG, e) for e in (0.01, 0.05, 0.1, 0.2, 0.5, 1.0)}
print(f"Volume I's package: {100 * PKG_P:.2f}% of the paths breach, and the next 1% sit just above the floor, "
      f"{np.sort(M_PKG)[5000:6000].mean():.2f} on average")
print("worst chance over the ball ($m radius):  " + "  ".join(f"{e:g}: {100 * q:.2f}%" for e, q in W_WORST.items()))

W_RADII = (0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0)
W_DESIGNS = {}
print("\nkeeping 5% over the ball: radius, line, delay, cost; under the law (fresh); the thinnest 5% average (needed)")
for e, b in zip(W_RADII, design_w(BOOK, W_RADII)):
    W_DESIGNS[e] = dict(design=b, p=BOOK.p(*b[:2]), p_fresh=FRESH.p(*b[:2]),
                        thin=thin_mean(thinnest(margin_parts(BOOK, b[0]), b[1])))
    w = W_DESIGNS[e]
    print(f"  {e:4g}  {b[0]:4.0f} {b[1]:6.3f} {b[2]:6.2f}   {100 * w['p']:5.2f}% ({100 * w['p_fresh']:.2f}%)   "
          f"{w['thin']:6.2f} ({e / ALPHA:.1f})")
for e in (0.1, 1.0):
    b = W_DESIGNS[e]["design"]
    print(f"the ball of ${e:g}m: its package at the records' corner breaches on {100 * CORNER.p(*b[:2]):.2f}% "
          f"(fresh {100 * CORNER_F.p(*b[:2]):.2f}%)")
W_MAXDEV = 1000.0 * float(np.mean(np.max(np.abs(BOOK.E), axis=1)))     # the paths' average largest deviation ($m)
W_SPREAD = (SIG_U / 50.0 - 1.0) * W_MAXDEV
BEST_THIN = thin_mean(thinnest(margin_parts(BOOK, 400.0), MAX_DELAY))
print(f"\nthe spread's doubt lies at {SIG_U / 50.0 - 1.0:.3f} x {W_MAXDEV:.1f} = {W_SPREAD:.1f}: the thinnest 5% would need "
      f"to average {W_SPREAD / ALPHA:.0f}; a $400m line and 7 quarters give {BEST_THIN:.1f}")

mw = np.sort(thinnest(margin_parts(BOOK, W_DESIGNS[0.1]["design"][0]), W_DESIGNS[0.1]["design"][1]))
x = 100.0 * (np.arange(N) + 0.5) / N
sel = slice(0, int(0.12 * N))
pushed = (x >= 5.0) & (x <= 100 * W_WORST[0.1])
mp = np.sort(M_PKG)
fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.axhline(0.0, color="#999999", lw=0.8)
ax.fill_between(x[pushed], 0.0, np.maximum(mp[pushed], 0.0), color="#B71C1C", alpha=0.3, lw=0,
                label=f"pushed below the floor by a radius of \\$0.1m: 5.0% to {100 * W_WORST[0.1]:.1f}%")
ax.plot(x[sel], mp[sel], color="#1F4E79", lw=1.8, label="Volume I's package")
ax.plot(x[sel], mw[sel], color="#E65100", lw=1.8, label="the package keeping 5% over that ball: \\$14.13m")
ax.set(xlim=(0, 12), ylim=(-32, 32), xlabel="paths, thinnest first (%)", ylabel="thinnest margin ($ million)",
       title="A ball of \\$0.1m lifts the chance from 5.0% to 7.1% (seed 26211)")
ax.legend(frameon=False, fontsize=8, loc="lower right"); plt.tight_layout(); plt.show()

## Panel 5 — Set a radius from a record (Theorem 11.3(iv); Proposition 11.2(iv); Figure 11.2(b); Worked Example 11.1, Step 4: designed on 1,000 paths, 38.5% keep 5% at \$10.19m; a one-point margin, 87.0% at \$12.09m; a radius of \$20,000, 86.5% at \$11.94m; of \$50,000, 97.5% at \$12.94m)

AXIOM Lab 2.11, step 4. A ball fits the sampling error of a record. As in Chapter 9's sample-size study (Online
Appendix 9.B.2), 200 records of 1,000 paths each (seeds 900000–900199) are drawn from the declared law. On each, a
package is designed two ways, among Chapter 9's lines (\$120–220 million on the \$5 million grid): aimed at a lower
target, $5\% - \delta$ (a margin), or kept at 5 percent over a Wasserstein ball around the record (a radius, the
bisection run with 30 halvings, as the book's engine runs it). Each design's true chance is read on the fresh 100,000
paths. A design fitted to its own small record looks better than it is (Proposition 11.2(iv)); a radius, like a
margin, buys reliability, and says what it protects against.

In [ ]:
CH9_LINES = np.arange(120.0, 220.01, 5.0)                  # Chapter 9's lines for the sample-size study
RECORDS = [Paths.draw(900000 + r, n=1000) for r in range(200)]
REC_MARGINS = (0.0, 0.005, 0.01, 0.015, 0.02)
REC_RADII = (0.01, 0.02, 0.03, 0.05, 0.1, 0.2)
rows_m = {mg: [] for mg in REC_MARGINS}
rows_w = {e: [] for e in REC_RADII}
for P in RECORDS:
    for mg in REC_MARGINS:
        b = cheapest(P, p=ALPHA - mg, lines=CH9_LINES)
        rows_m[mg].append((FRESH.p(*b[:2]), b[2], b[0]))
    for e, b in zip(REC_RADII, design_w(P, REC_RADII, lines=CH9_LINES, steps=30)):
        rows_w[e].append((FRESH.p(*b[:2]), b[2], b[0]))

def summary(rows):
    tp, cs, ln = (np.array(v) for v in zip(*rows))
    return dict(meet=float(np.mean(tp <= ALPHA)), mean_true=float(tp.mean()), q90=float(np.quantile(tp, 0.9)),
                mean_cost=float(cs.mean()), mean_line=float(ln.mean()), reps=len(tp))

REC_M = {mg: summary(r) for mg, r in rows_m.items()}
REC_W = {e: summary(r) for e, r in rows_w.items()}
print("designed on a record of 1,000 paths, judged on the fresh 100,000:")
print("                     keep 5%   mean true chance   90th percentile   mean cost")
for mg, s in REC_M.items():
    print(f"  target lower by {100 * mg:3.1f} pt {100 * s['meet']:6.1f}% {100 * s['mean_true']:12.2f}% "
          f"{100 * s['q90']:16.2f}% {s['mean_cost']:11.2f}")
for e, s in REC_W.items():
    print(f"  radius ${1e6 * e:7,.0f}    {100 * s['meet']:6.1f}% {100 * s['mean_true']:12.2f}% "
          f"{100 * s['q90']:16.2f}% {s['mean_cost']:11.2f}")
print(f"the book's 100,000 paths, Volume I's package: fresh {100 * PKG_F:.2f}% at {PKG[2]:.2f}")

fig, ax = plt.subplots(figsize=(7.6, 3.6))
sa = list(REC_M.values())
wa = [REC_M[0.0]] + list(REC_W.values())
ax.plot([s["mean_cost"] for s in sa], [100 * s["meet"] for s in sa], "o-", color="#1F4E79", lw=1.6, ms=4,
        label="lower the target (Chapter 9)")
ax.plot([s["mean_cost"] for s in wa], [100 * s["meet"] for s in wa], "s-", color="#E65100", lw=1.6, ms=4,
        label="a Wasserstein radius")
for mg in (0.005, 0.01, 0.015):
    s = REC_M[mg]
    ax.text(s["mean_cost"] + 0.12, 100 * s["meet"] - 4.0, f"{100 * mg:g} pt", fontsize=8, color="#1F4E79")
for e in (0.02, 0.05, 0.1):
    s = REC_W[e]
    ax.text(s["mean_cost"] - 0.12, 100 * s["meet"] + 3.0, f"\\${1000 * e:.0f}k", fontsize=8, color="#E65100", ha="right")
ax.annotate(f"no margin: {100 * REC_M[0.0]['meet']:.1f}%", xy=(REC_M[0.0]["mean_cost"], 100 * REC_M[0.0]["meet"]),
            xytext=(11.2, 36), fontsize=8, arrowprops=dict(arrowstyle="-", lw=0.6, color="#333333"))
ax.set(xlim=(10, 16), ylim=(30, 104), xlabel="average cost of the designs ($ million)",
       ylabel="designs keeping 5% (%)", title="On 1,000 paths a radius buys reliability like a margin (seed 26211)")
ax.legend(frameon=False, fontsize=8, loc="lower right"); plt.tight_layout(); plt.show()

## Panel 6 — Moments only (Proposition 11.1(iii)–(iv); Section 11.4: quarter 7's margin, \$147.5m on average with a spread of \$62.9m, breaches with chance up to 15.4%; 4.36 spreads of margin; a \$160m line and 6.06 quarters, \$53.29m)

AXIOM Lab 2.11, step 5. The widest family a record supports declares only means and spreads. For a margin of mean
$\mu > 0$ and spread $\sigma$, the worst chance that it falls to the floor is Cantelli's $\sigma^2/(\sigma^2 + \mu^2)$,
attained by a two-point law: the margin is exactly zero with that chance and $\mu + \sigma^2/\mu$ otherwise
(Proposition 11.1(iii)), the lump a closure produces. Every such law keeps 5 percent exactly when
$\mu \ge \sigma\sqrt{0.95/0.05} = 4.36\,\sigma$. With twelve quarterly margins, the worst chance that some margin falls
to zero lies between the largest bound and their sum (part (iv)). A delay adds $\ell_k\phi$ to each quarter's mean and
leaves its spread alone, so the cheapest package keeping every quarter's bound at 5 percent takes, at each line, the
delay $\max_k (4.36\,\sigma_k - \mu_k)^+/\ell_k$.

In [ ]:
def moments(P, line, phi):
    # the mean and spread of each quarter's margin above the floor ($m) under the package (line, phi)
    m = 1000.0 * (BASE[None, :] - (DEPTH0 - DPQ * phi) * SHAPE[None, :] + P.E - P.lost(line) - FLOOR)
    return m.mean(axis=0), m.std(axis=0)

def cantelli(mu, sd):
    # Proposition 11.1(iii): the worst chance of a margin at or below zero, over laws with mean mu and spread sd
    return np.where(mu > 0, sd ** 2 / (sd ** 2 + mu ** 2), 1.0)

MU, SD = moments(BOOK, *PKG[:2])
CB = cantelli(MU, SD)
k7 = int(np.argmax(CB))
print("Volume I's package, quarter by quarter:")
print("  quarter          " + " ".join(f"{k:6d}" for k in range(1, Q + 1)))
print("  mean margin      " + " ".join(f"{x:6.1f}" for x in MU))
print("  spread           " + " ".join(f"{x:6.1f}" for x in SD))
print("  Cantelli (%)     " + " ".join(f"{100 * x:6.1f}" for x in CB))
print(f"worst in quarter {k7 + 1}: {100 * CB[k7]:.1f}%, a two-point law: the margin is 0 with that chance, "
      f"{MU[k7] + SD[k7] ** 2 / MU[k7]:.1f} otherwise; the bounds sum to {100 * CB.sum():.1f}%, so the union bound says nothing")

Z = math.sqrt((1 - ALPHA) / ALPHA)
Z_UNION = math.sqrt((1 - ALPHA / Q) / (ALPHA / Q))
best = None
for line in LINES:
    mu0, sd = moments(BOOK, line, 0.0)
    phi = max(0.0, float(np.max((Z * sd - mu0) / LIFT)))
    if phi <= MAX_DELAY and (best is None or cost(line, phi) < best[2]):
        best = (float(line), phi, float(cost(line, phi)))
MOM = best
MOM_MU0, MOM_SD = moments(BOOK, MOM[0], 0.0)
print(f"\nevery quarter's bound at 5% (mean at least {Z:.3f} spreads): line {MOM[0]:.0f}, delay {MOM[1]:.3f}, cost {MOM[2]:.2f}; "
      f"under the law {100 * BOOK.p(*MOM[:2]):.3f}% (fresh {100 * FRESH.p(*MOM[:2]):.3f}%)")
print(f"the program's appetite by the union bound would need {Z_UNION:.2f} spreads in every quarter")
print(f"at the records' corner the moment package breaches on {100 * CORNER.p(*MOM[:2]):.2f}% "
      f"(fresh {100 * CORNER_F.p(*MOM[:2]):.2f}%)")

## Panel 7 — Price the doubt (Worked Example 11.1, Step 5; Table 11.3; Figure 11.3(b): \$24.00m against \$10.44m, a price of ambiguity of \$13.56m, mostly the spread's; a record twice as long worth \$4.63m)

Worked Example 11.1, Step 5. Every declaration's package side by side: its cost, its chance of a breach under
Chapter 9's law, and under the records' worst law (closures 0.315 a year, spread \$61.6 million), on the book's
100,000 paths and on the fresh ones. The **price of ambiguity** is what keeping the appetite under every law the
records allow costs beyond trusting the law; the **value of a longer record** is what a record twice as long, at the
same estimates, would save. Each chance is a count of 100,000 paths, shown exactly, to three decimals; Table 11.3
rounds them to one.

In [ ]:
T113 = [("One law (Chapter 9)", PKG), ("A set, max-min choice (Chapter 10)", CH10 + (cost(*CH10),)),
        ("Wasserstein ball, $0.1 million", W_DESIGNS[0.1]["design"]), ("Divergence ball, d = 0.01", KL_DESIGNS[0.01]["design"]),
        ("Closures' doubt (rate <= 0.315)", CASES["rate"]["design"]), ("Spread's doubt (spread <= 61.6)", CASES["spread"]["design"]),
        ("The records' family (both)", CASES["both"]["design"]), ("Records twice as long", CASES["longer"]["design"]),
        ("Moments only (every quarter)", MOM)]
TABLE = []
print(f"{'declaration':36s} {'line':>5s} {'delay':>6s} {'cost':>6s} | {'law':>7s} {'fresh':>7s} | {'worst law':>9s} {'fresh':>7s}")
for name, (l_, ph, co) in T113:
    row = dict(name=name, line=l_, delay=ph, cost=co, p_law=BOOK.p(l_, ph), p_law_fresh=FRESH.p(l_, ph),
               p_worst=CORNER.p(l_, ph), p_worst_fresh=CORNER_F.p(l_, ph))
    TABLE.append(row)
    print(f"{name:36s} {l_:5.1f} {ph:6.3f} {co:6.2f} | {100 * row['p_law']:6.3f}% {100 * row['p_law_fresh']:6.3f}% | "
          f"{100 * row['p_worst']:8.3f}% {100 * row['p_worst_fresh']:6.3f}%")
PRICE = CASES["both"]["design"][2] - PKG[2]
RECORD_VALUE = CASES["both"]["design"][2] - CASES["longer"]["design"][2]
gaps = [abs(r[a] - r[b]) for r in TABLE for a, b in (("p_law", "p_law_fresh"), ("p_worst", "p_worst_fresh"))]
SE_MAX = max(math.sqrt(r[k] * (1 - r[k]) / N) for r in TABLE for k in ("p_law", "p_worst"))
print(f"\nthe price of ambiguity: {CASES['both']['design'][2]:.2f} - {PKG[2]:.2f} = {PRICE:.2f}, of which the spread's doubt "
      f"alone accounts for {CASES['spread']['design'][2] - PKG[2]:.2f}")
print(f"the value of a record twice as long: {CASES['both']['design'][2]:.2f} - {CASES['longer']['design'][2]:.2f} = "
      f"{RECORD_VALUE:.2f}")
print(f"fresh paths within {100 * max(gaps):.2f} points of the book's; the largest standard error of a chance in the table: "
      f"{100 * SE_MAX:.3f} points")

## Validation — the book's numbers, and agreement with `DCT_V2_Ch11_Lab.xlsx`

Each checkpoint is a number printed in Chapter 11 (Sections 11.1–11.5, Worked Example 11.1, Table 11.3, Figures
11.1–11.3) or in Online Appendix Example 11.B.2, with a tolerance of half a unit in its last printed digit (counts that
fall exactly halfway, such as 7,665 paths of 100,000, pass either way); a few check what a proposition promises.
Chances are in percent. The divergence ball's package at the records' corner (10,248 paths of 100,000) is checked
against Online Appendix Example 11.B.2's two decimals, and the levers' floor, $\mathrm{kl}(0.05\,\|\,10^{-5}) =
0.3771$, against the Online Appendix's 0.377. The workbook recomputes the checkpoints marked *workbook* with Excel formulas
and shows PASS against the same engine values on its `Checks` sheet.

In [ ]:
pc = lambda x: 100 * x
h = lambda dp: 0.5 * 10.0 ** -dp + 1e-9          # half a unit in the last printed digit
OA_RATE = (4.47, 4.57, 4.73, 5.00, 5.27, 5.53, 5.80, 6.16, 6.35)
OA_SPREAD = (1.03, 1.59, 2.52, 5.00, 8.59, 12.92, 14.43, 17.87)
OA_CASES = {"rate": (6.16, 6.06, 210, 0.667, 11.64, 4.88, 4.63, 4.53), "spread": (14.43, 14.44, 170, 2.218, 22.85, 4.91, 1.06, 1.03),
            "both": (15.79, 15.83, 200, 2.250, 24.00, 4.91, 0.89, 0.86), "longer": (11.60, 11.43, 195, 1.690, 19.37, 4.87, 1.66, 1.62)}
OA_KL_WORST = {0.001: 6.00, 0.003: 6.78, 0.01: 8.36, 0.03: 11.15, 0.1: 17.29, 0.3: 28.85, 1.0: 55.55}
OA_KL = {0.001: (4.08, 160, 0.924, 12.19, 4.05), 0.002: (3.74, 165, 0.998, 12.93, 3.68), 0.005: (3.11, 170, 1.162, 14.40, 3.09),
         0.01: (2.50, 175, 1.354, 16.08, 2.51), 0.02: (1.78, 180, 1.671, 18.77, 1.75), 0.03: (1.34, 175, 1.955, 20.89, 1.30),
         0.05: (0.81, 195, 2.356, 24.70, 0.79), 0.075: (0.46, 215, 2.737, 28.35, 0.46), 0.1: (0.27, 200, 3.173, 31.39, 0.30),
         0.15: (0.10, 240, 3.851, 38.01, 0.11)}
OA_W_WORST = {0.01: 5.64, 0.05: 6.47, 0.1: 7.11, 0.2: 8.05, 0.5: 10.00, 1.0: 12.32}
OA_W = {0.01: (165, 0.827, 11.57, 4.40, 4.32), 0.02: (165, 0.887, 12.04, 4.17, 4.11), 0.05: (165, 1.007, 13.00, 3.71, 3.66),
        0.1: (165, 1.148, 14.13, 3.23, 3.21), 0.2: (170, 1.335, 15.78, 2.61, 2.61), 0.5: (175, 1.761, 19.34, 1.67, 1.61),
        1.0: (175, 2.337, 23.95, 0.91, 0.89)}
OA_REC_M = {0.0: (38.5, 5.21, 6.23, 10.19), 0.005: (70.0, 4.70, 5.77, 11.09), 0.01: (87.0, 4.21, 5.09, 12.09),
            0.015: (98.5, 3.73, 4.52, 13.12), 0.02: (100.0, 3.22, 3.85, 14.33)}
OA_REC_W = {0.01: (78.5, 4.51, 5.51, 11.44), 0.02: (86.5, 4.26, 5.19, 11.94), 0.03: (91.5, 4.08, 4.96, 12.33),
            0.05: (97.5, 3.80, 4.60, 12.94), 0.1: (99.5, 3.32, 4.06, 14.07), 0.2: (100.0, 2.70, 3.31, 15.73)}
OA_MU = (229.0, 198.6, 176.9, 162.1, 152.9, 148.4, 147.5, 149.5, 153.3, 158.9, 166.1, 174.0)
OA_SD = (50.1, 58.4, 61.2, 62.3, 62.7, 62.9, 62.9, 62.9, 63.1, 63.1, 63.3, 63.5)
OA_CB = (4.6, 8.0, 10.7, 12.9, 14.4, 15.2, 15.4, 15.0, 14.5, 13.6, 12.7, 11.8)
OA_CORNER = {"Chapter 10's max-min package": (CH10, 16.16, 16.05), "the closures' doubt's package": (CASES["rate"]["design"], 14.43, 14.39),
             "the spread's doubt's package": (CASES["spread"]["design"], 5.86, 5.75),
             "twice the record's package": (CASES["longer"]["design"], 7.67, 7.44),
             "the divergence ball's package, d = 0.01": (KL_DESIGNS[0.01]["design"], 10.25, 10.08),
             "the divergence ball's package, d = 0.03": (KL_DESIGNS[0.03]["design"], 6.87, 6.71),
             "the Wasserstein ball's package, $0.1m": (W_DESIGNS[0.1]["design"], 12.15, 12.05),
             "the Wasserstein ball's package, $1m": (W_DESIGNS[1.0]["design"], 5.23, 5.15), "the moment package": (MOM, 0.35, 0.37)}

checks = [
    # Proposition 11.2; Section 11.1; Worked Example 11.1, Step 1 (Panel 1)
    ("records: rate, upper 95% bound (workbook)", LAM_U, 0.3148, h(4)),
    ("records: rate, lower 95% bound (workbook)", LAM_L, 0.0178, h(4)),
    ("records: rate bound, Section 11.1 (workbook)", LAM_U, 0.315, h(3)),
    ("records: spread, upper 95% bound (workbook)", SIG_U, 61.60, h(2)),
    ("records: spread, lower 95% bound (workbook)", SIG_L, 42.27, h(2)),
    ("twice the record: rate bound (workbook)", LAM_U2, 0.2288, h(4)),
    ("twice the record: spread, upper bound (workbook)", SIG_U2, 57.60, h(2)),
    ("twice the record: spread, lower bound (workbook)", SIG_L2, 44.28, h(2)),
    ("the records' family: confidence together, % (workbook)", 100 * (1 - 0.05 - 0.05), 90, 1e-9),
    ("Volume I's package: line", PKG[0], 170, 0),
    ("Volume I's package: delay", PKG[1], 0.668, h(3)),
    ("Volume I's package: cost (workbook)", PKG[2], 10.44, h(2)),
    ("Volume I's package: chance under the law", pc(PKG_P), 5.00, h(2)),
    ("Volume I's package: chance on the fresh paths", pc(PKG_F), 4.91, h(2)),
] + [(f"Volume I's package at rate {lam:.3f}", pc(p_), b_, h(2)) for lam, p_, b_ in zip(RATES, BY_RATE, OA_RATE)] + [
    (f"Volume I's package at spread {s:.1f}", pc(p_), b_, h(2)) for s, p_, b_ in zip(SPREADS, BY_SPREAD, OA_SPREAD)] + [
    ("Step 1: across the closures' range, from (%)", pc(min(BY_RATE[:8])), 4.5, h(1)),
    ("Step 1: across the closures' range, to (%)", pc(max(BY_RATE[:8])), 6.2, h(1)),
    ("Step 1: across the spread's range, from (%)", pc(BY_SPREAD[1]), 1.6, h(1)),
    ("Step 1: across the spread's range, to (%)", pc(BY_SPREAD[6]), 14.4, h(1)),
    ("Volume I's package at the records' corner", pc(P_CORNER), 15.79, h(2)),
    ("Volume I's package at the corner, fresh paths", pc(P_CORNER_F), 15.83, h(2)),
    ("Section 11.1: at the corner, %", pc(P_CORNER), 15.8, h(1)),
    ("Proposition 11.2(ii): the spread's bound holds, % of simulated records", pc(COVER["spread"]), 95.0, 0.25),
    ("Proposition 11.2(iii): both bounds hold in at least 90% of records", int(COVER["both"] >= 0.90), 1, 0),
]
for key, (pp, ppf, l_, ph, co, pf, pl, plf) in OA_CASES.items():
    c = CASES[key]
    checks += [(f"{NAMES[key]}: Volume I's package at its law", pc(c["p_pkg"]), pp, h(2)),
               (f"{NAMES[key]}: Volume I's package, fresh", pc(c["p_pkg_fresh"]), ppf, h(2)),
               (f"{NAMES[key]}: line", c["design"][0], l_, 0),
               (f"{NAMES[key]}: delay", c["design"][1], ph, h(3)),
               (f"{NAMES[key]}: cost (workbook)", c["design"][2], co, h(2)),
               (f"{NAMES[key]}: fresh paths at its law", pc(c["p_fresh"]), pf, h(2)),
               (f"{NAMES[key]}: under the declared law", pc(c["p_law"]), pl, h(2)),
               (f"{NAMES[key]}: under the declared law, fresh", pc(c["p_law_fresh"]), plf, h(2))]
checks += [
    # Theorem 11.1(iv); Worked Example 11.1, Step 2 (Panel 2)
    ("the family's package: largest chance on the 49-law grid (workbook)", pc(worst_law[2]), 5.00, h(2)),
    ("the family's package: the grid's worst law is the corner", int(worst_law[0] == GRID[-1][0] and worst_law[1] == GRID[-1][1]), 1, 0),
    ("Step 2: on fresh paths at the corner (%)", pc(CASES["both"]["p_fresh"]), 4.9, h(1)),
    ("Step 2: under the declared law (%)", pc(CASES["both"]["p_law"]), 0.9, h(1)),
    # Theorem 11.2; Figure 11.1; Section 11.2; Step 3 (Panel 3)
    ("divergence: radius of the closures' doubt (workbook)", RADII["closures' doubt"], 0.439, h(3)),
    ("divergence: radius of the spread's doubt (workbook)", RADII["spread's doubt"], 0.603, h(3)),
    ("divergence: radius of both (workbook)", RADII["both doubts"], 1.042, h(3)),
    ("divergence: the law's appetite at 0.439, per million (workbook)", 1e6 * appetite(RADII["closures' doubt"]), 2.93, h(2)),
    ("divergence: the law's appetite at 0.603, per 10 million (workbook)", 1e7 * appetite(RADII["spread's doubt"]), 1.09, h(2)),
    ("divergence: the law's appetite at 1.042, per 100 billion (workbook)", 1e11 * appetite(RADII["both doubts"]), 1.69, h(2)),
    ("Section 11.2: 3 in a million", round(1e6 * appetite(RADII["closures' doubt"])), 3, 0),
    ("Step 3: 2 in 100 billion", round(1e11 * appetite(RADII["both doubts"])), 2, 0),
    ("the levers' floor: paths that breach at a $400m line and 7 quarters", P_MIN * N, 1, 0),
    ("the levers' floor: kl(0.05 || 1e-5), Online Appendix 11.B.2 (workbook)", D_LEVERS, 0.377, h(3)),
] + [(f"divergence: worst chance of Volume I's package at d = {d:g} (workbook)", pc(KL_WORST[d]), b_, h(2))
     for d, b_ in OA_KL_WORST.items()]
for d, (a_, l_, ph, co, pf) in OA_KL.items():
    k_ = KL_DESIGNS[d]
    checks += [(f"divergence d = {d:g}: the law's appetite, % (workbook)", pc(k_["alpha"]), a_, h(2)),
               (f"divergence d = {d:g}: line", k_["design"][0], l_, 0),
               (f"divergence d = {d:g}: delay", k_["design"][1], ph, h(3)),
               (f"divergence d = {d:g}: cost (workbook)", k_["design"][2], co, h(2)),
               (f"divergence d = {d:g}: fresh paths", pc(k_["p_fresh"]), pf, h(2))]
checks += [
    ("Table 11.3: divergence ball d = 0.01, under the law (%)", pc(KL_DESIGNS[0.01]["p"]), 2.5, h(1)),
    # Theorem 11.3; Figure 11.2(a); Section 11.3; Step 3 (Panel 4)
] + [(f"Wasserstein: worst chance of Volume I's package at ${e:g}m", pc(W_WORST[e]), b_, h(2)) for e, b_ in OA_W_WORST.items()]
for e, (l_, ph, co, pl, pf) in OA_W.items():
    w_ = W_DESIGNS[e]
    checks += [(f"Wasserstein ${e:g}m: line", w_["design"][0], l_, 0),
               (f"Wasserstein ${e:g}m: delay", w_["design"][1], ph, h(3)),
               (f"Wasserstein ${e:g}m: cost (workbook)", w_["design"][2], co, h(2)),
               (f"Wasserstein ${e:g}m: under the law", pc(w_["p"]), pl, h(2)),
               (f"Wasserstein ${e:g}m: fresh paths", pc(w_["p_fresh"]), pf, h(2)),
               (f"Wasserstein ${e:g}m: the thinnest 5% average radius / 0.05 (workbook)", w_["thin"], e / ALPHA, 1e-6)]
checks += [
    ("Section 11.3: thinnest 5% must average, $m (workbook)", W_DESIGNS[0.1]["thin"], 2, h(0)),
    ("Wasserstein: the spread's excess (workbook)", SIG_U / 50.0 - 1.0, 0.232, h(3)),
    ("Wasserstein: the paths' average largest deviation, $m", W_MAXDEV, 113.2, h(1)),
    ("Wasserstein: radius of the spread's doubt, $m (workbook)", W_SPREAD, 26.3, h(1)),
    ("Wasserstein: the thinnest 5% would need, $m (workbook)", W_SPREAD / ALPHA, 525, h(0)),
    ("Wasserstein: within the levers' reach, $m", BEST_THIN, 121.9, h(1)),
    ("Section 11.3: the levers reach, $m", BEST_THIN, 122, h(0)),
]
for mg, (mt, mtrue, q9, mc) in OA_REC_M.items():
    s = REC_M[mg]
    checks += [(f"records of 1,000, target lower by {100 * mg:g} pt: keep 5%", pc(s["meet"]), mt, h(1)),
               (f"records of 1,000, target lower by {100 * mg:g} pt: mean true chance", pc(s["mean_true"]), mtrue, h(2)),
               (f"records of 1,000, target lower by {100 * mg:g} pt: 90th percentile", pc(s["q90"]), q9, h(2)),
               (f"records of 1,000, target lower by {100 * mg:g} pt: mean cost", s["mean_cost"], mc, h(2))]
for e, (mt, mtrue, q9, mc) in OA_REC_W.items():
    s = REC_W[e]
    checks += [(f"records of 1,000, radius ${1e6 * e:,.0f}: keep 5%", pc(s["meet"]), mt, h(1)),
               (f"records of 1,000, radius ${1e6 * e:,.0f}: mean true chance", pc(s["mean_true"]), mtrue, h(2)),
               (f"records of 1,000, radius ${1e6 * e:,.0f}: 90th percentile", pc(s["q90"]), q9, h(2)),
               (f"records of 1,000, radius ${1e6 * e:,.0f}: mean cost", s["mean_cost"], mc, h(2))]
checks += [(f"moments: mean margin, quarter {k + 1}", MU[k], OA_MU[k], h(1)) for k in range(Q)]
checks += [(f"moments: spread, quarter {k + 1}", SD[k], OA_SD[k], h(1)) for k in range(Q)]
checks += [(f"moments: Cantelli bound, quarter {k + 1} (workbook)", pc(CB[k]), OA_CB[k], h(1)) for k in range(Q)]
checks += [
    ("moments: the worst quarter (workbook)", k7 + 1, 7, 0),
    ("moments: the bounds' sum, % (workbook)", pc(CB.sum()), 148.6, h(1)),
    ("moments: spreads of margin at 5% (workbook)", Z, 4.359, h(3)),
    ("moments: spreads of margin, Section 11.4 (workbook)", Z, 4.36, h(2)),
    ("moments: the union bound would need, spreads (workbook)", Z_UNION, 15.46, h(2)),
    ("moments: line (workbook)", MOM[0], 160, 0),
    ("moments: delay (workbook)", MOM[1], 6.062, h(3)),
    ("moments: cost (workbook)", MOM[2], 53.29, h(2)),
    ("moments: chance under the law, %", pc(BOOK.p(*MOM[:2])), 0.026, h(3)),
]
for name, (pk, pb, pf) in OA_CORNER.items():
    checks += [(f"at the corner: {name}", pc(CORNER.p(*pk[:2])), pb, h(2)),
               (f"at the corner: {name}, fresh", pc(CORNER_F.p(*pk[:2])), pf, h(2))]
checks += [
    ("Chapter 10's max-min package: line", CH10[0], 231.6, h(1)),
    ("Chapter 10's max-min package: delay", CH10[1], 0.449, h(3)),
    ("Chapter 10's max-min package: cost (workbook)", cost(*CH10), 10.54, h(2)),
    ("Chapter 10's max-min package: chance under the law", pc(BOOK.p(*CH10)), 5.64, h(2)),
    # Table 11.3 and Step 5 (Panel 7)
    ("price of ambiguity (workbook)", PRICE, 13.56, h(2)),
    ("value of a record twice as long (workbook)", RECORD_VALUE, 4.63, h(2)),
    ("Table 11.3: chances on fresh paths within 0.3 points", int(max(gaps) <= 0.003), 1, 0),
    ("Section 11.5: every chance in Table 11.3 has a standard error below 0.12 points (workbook)", int(SE_MAX < 0.0012), 1, 0),
]
T113_BOOK = {"One law (Chapter 9)": (170, 0.67, 10.44, 5.0, 15.8), "A set, max-min choice (Chapter 10)": (232, 0.45, 10.54, 5.6, 16.2),
             "Wasserstein ball, $0.1 million": (165, 1.15, 14.13, 3.2, 12.2), "Divergence ball, d = 0.01": (175, 1.35, 16.08, 2.5, None),
             "Closures' doubt (rate <= 0.315)": (210, 0.67, 11.64, 4.6, 14.4), "Spread's doubt (spread <= 61.6)": (170, 2.22, 22.85, 1.1, 5.9),
             "The records' family (both)": (200, 2.25, 24.00, 0.9, None), "Records twice as long": (195, 1.69, 19.37, 1.7, 7.7),
             "Moments only (every quarter)": (160, 6.06, 53.29, 0.03, 0.4)}
for r in TABLE:
    l_, ph, co, pl, pw = T113_BOOK[r["name"]]
    checks += [(f"Table 11.3, {r['name']}: line", r["line"], l_, h(0)), (f"Table 11.3, {r['name']}: delay", r["delay"], ph, h(2)),
               (f"Table 11.3, {r['name']}: cost (workbook)", r["cost"], co, h(2)),
               (f"Table 11.3, {r['name']}: under the law", pc(r["p_law"]), pl, h(2 if pl < 0.1 else 1))]
    if pw is not None:
        checks.append((f"Table 11.3, {r['name']}: under the worst law", pc(r["p_worst"]), pw, h(1)))
checks += [("Table 11.3, the records' family: under the worst law (the grid's corner)", pc(worst_law[2]), 5.0, h(1))]
ok = 0
for name, got, want, tol in checks:
    status = "PASS" if abs(got - want) <= tol else "FAIL"; ok += status == "PASS"
    print(f"{name:78s} {got:12.4f}   book {want:10.4f}   {status}")
print(f"\n{ok} of {len(checks)} checkpoints PASS")

**Next.** Chapter 12, *Multi-Objective Enterprise Optimization*, opens Part IV. Table 11.3 already sets two objectives
side by side, a package's cost and its protection under the records' worst law; Chapter 12 treats such trade-offs
directly, with Pareto frontiers and the prices along them. AXIOM-11 runs the same models interactively (Lab 2.11);
this notebook and the workbook are its reproducible record.